# Panel compatibility

Fast, interpretable checks for adding probes to an existing panel.

In [ ]:
#| default_exp panel_compatibility

In [ ]:
#| export
from __future__ import annotations

from collections import defaultdict
import re

import pandas as pd

from probeharbour.thermo import ThermoParameters

In [ ]:
#| export
def reverse_complement(sequence: str) -> str:
    """Return the uppercase reverse complement of an A/C/G/T sequence."""
    return str(sequence).upper().translate(str.maketrans("ACGT", "TGCA"))[::-1]


def _longest_common_substring(left: str, right: str) -> int:
    previous = [0] * (len(right) + 1)
    longest = 0
    for left_base in left:
        current = [0]
        for index, right_base in enumerate(right, start=1):
            value = previous[index - 1] + 1 if left_base == right_base else 0
            current.append(value)
            longest = max(longest, value)
        previous = current
    return longest


def _first_column(frame: pd.DataFrame, *names: str) -> str | None:
    return next((name for name in names if name in frame), None)


def _panel_frame(panel) -> pd.DataFrame:
    frame = panel.obs if hasattr(panel, "obs") else panel
    if not isinstance(frame, pd.DataFrame):
        raise TypeError("panel must be a ProbeStore or pandas DataFrame")
    missing = {"rbd5", "rbd3"} - set(frame.columns)
    if missing:
        raise ValueError(f"Panel is missing required columns: {sorted(missing)}")
    result = frame.copy()
    id_column = _first_column(result, "panel_record_id", "candidate_id", "probe_name")
    display_column = _first_column(result, "candidate_id", "probe_name", "panel_record_id")
    result["_panel_id"] = result[id_column].astype(str) if id_column else [f"probe-{i + 1}" for i in range(len(result))]
    result["_display_id"] = result[display_column].astype(str) if display_column else result["_panel_id"]
    result["_binder"] = result["rbd5"].astype(str).str.upper() + result["rbd3"].astype(str).str.upper()
    result["_gene"] = result[_first_column(result, "gene", "gene_name", "gene_id")].astype(str) if _first_column(result, "gene", "gene_name", "gene_id") else ""
    return result


def _genomic_blocks(row):
    """Return chromosome and physically bound blocks, never the spanned intron."""
    packed = row.get("coordinates")
    packed = packed if isinstance(packed, dict) else {}
    def field(name, packed_name=None):
        value = row.get(name)
        return packed.get(packed_name or name) if value is None or pd.isna(value) else value

    chrom = field("chrom")
    if chrom is None or pd.isna(chrom):
        return None
    blocks = []
    for prefix in ("rbd5", "rbd3"):
        start = field(f"{prefix}_chrom_start", f"{prefix}_start")
        end = field(f"{prefix}_chrom_end", f"{prefix}_end")
        if start is not None and end is not None and pd.notna(start) and pd.notna(end):
            blocks.append((int(start), int(end)))
    if not blocks:
        start, end = field("chrom_start", "start"), field("chrom_end", "end")
        sizes, offsets = row.get("bed_block_sizes"), row.get("bed_block_starts")
        if start is not None and end is not None and pd.notna(start) and pd.notna(end):
            if sizes is not None and offsets is not None and pd.notna(sizes) and pd.notna(offsets):
                parsed_sizes = [int(value) for value in str(sizes).split(",") if value]
                parsed_offsets = [int(value) for value in str(offsets).split(",") if value]
                blocks = [(int(start) + offset, int(start) + offset + size)
                          for size, offset in zip(parsed_sizes, parsed_offsets)]
            else:
                blocks = [(int(start), int(end))]
    return (str(chrom).removeprefix("chr"), blocks) if blocks else None


def _block_distance(left, right):
    if left is None or right is None or left[0] != right[0]:
        return None
    return min(max(0, b_start - a_end, a_start - b_end)
               for a_start, a_end in left[1] for b_start, b_end in right[1])

In [ ]:
#| export
def check_panel_compatibility(
    query,
    reference,
    *,
    complement_run_threshold: int = 12,
    heterodimer_dg_threshold: float = -10.5,
    seed_length: int = 6,
    heterodimer_top_n: int = 5,
    calculate_heterodimer: bool = True,
    fail_on_target_overlap: bool = False,
    min_distance: int = 75,
    exclude_same_id: bool = False,
    parameters: ThermoParameters = ThermoParameters(),
) -> pd.DataFrame:
    """Compare new probes with an existing panel and return one row per new probe.

    Primer3 is evaluated only for the closest ``heterodimer_top_n`` sequence
    matches per query. This keeps panel-scale checks interactive while retaining
    the raw worst ΔG and the identity of the corresponding reference probe.
    """
    if complement_run_threshold < 1 or seed_length < 1 or heterodimer_top_n < 1:
        raise ValueError("run threshold, seed length, and heterodimer_top_n must be positive")
    new = _panel_frame(query)
    old = _panel_frame(reference)

    reference_records = list(old[["_panel_id", "_display_id", "_binder", "rbd5", "rbd3", "_gene"]].itertuples(index=False, name=None))

    reference_intervals = [_genomic_blocks(row) for _, row in old.iterrows()]
    query_intervals = {
        str(row["_panel_id"]): _genomic_blocks(row) for _, row in new.iterrows()
    }
    binder_index = defaultdict(list)
    arm_index = defaultdict(list)
    kmer_index = defaultdict(set)
    for index, (panel_id, display_id, binder, rbd5, rbd3, gene) in enumerate(reference_records):
        binder_index[binder].append(index)
        arm_index[str(rbd5).upper()].append(index)
        arm_index[str(rbd3).upper()].append(index)
        complement = reverse_complement(binder)
        for start in range(max(0, len(complement) - seed_length + 1)):
            kmer_index[complement[start:start + seed_length]].add(index)

    reference_genes = {gene for *_, gene in reference_records if gene and gene.lower() != "nan"}
    structure_kwargs = parameters.as_structure_primer3()
    primer3 = None
    if calculate_heterodimer:
        try:
            import primer3 as primer3_module
        except ImportError as exc:
            raise RuntimeError("Primer3 support requires the optional primer3-py dependency") from exc
        primer3 = primer3_module

    rows = []
    query_records = new[["_panel_id", "_binder", "rbd5", "rbd3", "_gene"]].itertuples(index=False, name=None)
    for query_id, binder, query_rbd5, query_rbd3, gene in query_records:
        query_arms = {str(query_rbd5).upper(), str(query_rbd3).upper()}
        candidate_indices = set(binder_index.get(binder, ())) | set(binder_index.get(reverse_complement(binder), ()))
        for arm in query_arms:
            candidate_indices.update(arm_index.get(arm, ()))
        if len(binder) >= seed_length:
            for start in range(len(binder) - seed_length + 1):
                candidate_indices.update(kmer_index.get(binder[start:start + seed_length], ()))

        comparisons = []
        for index in candidate_indices:
            panel_id, display_id, reference_binder, reference_rbd5, reference_rbd3, reference_gene = reference_records[index]
            if exclude_same_id and panel_id == query_id:
                continue
            run = _longest_common_substring(binder, reverse_complement(reference_binder))
            comparisons.append((run, display_id, reference_binder, reference_rbd5, reference_rbd3))
        comparisons.sort(key=lambda item: (-item[0], item[1]))

        exact = [reference_records[i][1] for i in binder_index.get(binder, ())
                 if not exclude_same_id or reference_records[i][0] != query_id]
        reverse_exact = [reference_records[i][1] for i in binder_index.get(reverse_complement(binder), ())
                         if not exclude_same_id or reference_records[i][0] != query_id]
        duplicate_arm_ids = sorted({reference_records[i][1] for arm in query_arms for i in arm_index.get(arm, ())
                                    if not exclude_same_id or reference_records[i][0] != query_id})
        longest_run = comparisons[0][0] if comparisons else 0
        closest_probe = comparisons[0][1] if comparisons else None

        worst_dg = None
        worst_dg_probe = None
        if primer3 is not None:
            for _, panel_id, reference_binder, _, _ in comparisons[:heterodimer_top_n]:
                dg = round(float(primer3.calc_heterodimer(binder, reference_binder, **structure_kwargs).dg) / 1000, 3)
                if worst_dg is None or dg < worst_dg:
                    worst_dg, worst_dg_probe = dg, panel_id

        target_overlap = bool(gene and gene.lower() != "nan" and gene in reference_genes)
        query_interval = query_intervals.get(str(query_id))
        genomic_neighbours = []
        if query_interval is not None:
            for index, reference_interval in enumerate(reference_intervals):
                distance = _block_distance(query_interval, reference_interval)
                if distance is not None:
                    genomic_neighbours.append((distance, reference_records[index][1]))
        genomic_neighbours.sort(key=lambda value: (value[0], value[1]))
        nearest_distance, nearest_name = genomic_neighbours[0] if genomic_neighbours else (None, None)
        reasons = []
        if exact: reasons.append("duplicate_binder")
        if reverse_exact: reasons.append("reverse_complement_binder")
        if duplicate_arm_ids: reasons.append("duplicate_arm")
        if longest_run >= complement_run_threshold: reasons.append("long_complementary_run")
        if worst_dg is not None and worst_dg < heterodimer_dg_threshold: reasons.append("strong_panel_heterodimer")
        if target_overlap and fail_on_target_overlap: reasons.append("target_already_present")
        if nearest_distance is not None and nearest_distance < min_distance:
            reasons.append("existing_panel_spacing")

        rows.append({
            "candidate_id": query_id,
            "closest_panel_probe": closest_probe,
            "longest_panel_complement": longest_run,
            "duplicate_binder": bool(exact),
            "reverse_complement_binder": bool(reverse_exact),
            "duplicate_arm": bool(duplicate_arm_ids),
            "duplicate_panel_probe": exact[0] if exact else (reverse_exact[0] if reverse_exact else None),
            "duplicate_arm_panel_probe": duplicate_arm_ids[0] if duplicate_arm_ids else None,
            "worst_panel_heterodimer_dg": worst_dg,
            "worst_panel_heterodimer_probe": worst_dg_probe,
            "target_gene_already_present": target_overlap,
            "nearest_existing_panel_probe_name": nearest_name,
            "nearest_existing_panel_probe_distance": nearest_distance,
            "existing_panel_spacing_violation": bool(nearest_distance is not None and nearest_distance < min_distance),
            "panel_compatibility_pass": not reasons,
            "panel_compatibility_reasons": reasons,
        })
    return pd.DataFrame(rows)

In [ ]:
#| export
def nearest_probe_distances(panel, *, min_distance: int = 75, role_column: str = "panel_role") -> pd.DataFrame:
    """Return self-excluding nearest genomic neighbours overall and by panel role.

    Distances are interval-to-interval base-pair gaps: overlapping intervals have
    distance zero. Rows without usable genomic coordinates retain missing evidence.
    """
    if min_distance < 0:
        raise ValueError("min_distance must be non-negative")
    frame = _panel_frame(panel)
    packed = [_genomic_blocks(row) for _, row in frame.iterrows()]

    roles = frame[role_column].fillna("unassigned").astype(str).tolist() if role_column in frame else ["unassigned"] * len(frame)
    ids = frame["_panel_id"].tolist()
    display_ids = frame["_display_id"].tolist()
    safe_roles = {role: re.sub(r"[^0-9a-zA-Z]+", "_", role).strip("_").lower() or "unassigned" for role in sorted(set(roles))}
    records = []
    for index, coordinates in enumerate(packed):
        candidates = []
        if coordinates is not None:
            for other_index, other in enumerate(packed):
                if other_index == index:
                    continue
                distance = _block_distance(coordinates, other)
                if distance is not None:
                    candidates.append((distance, str(display_ids[other_index]), other_index))
        candidates.sort()
        nearest = candidates[0] if candidates else None
        result = {
            "panel_record_id": ids[index],
            "nearest_probe_name": nearest[1] if nearest else None,
            "nearest_probe_distance": nearest[0] if nearest else float("nan"),
            "nearest_probe_role": roles[nearest[2]] if nearest else None,
            "panel_spacing_violation": bool(nearest and nearest[0] < min_distance),
            "pass_panel_spacing": not bool(nearest and nearest[0] < min_distance),
        }
        for role, safe_role in safe_roles.items():
            role_candidates = [item for item in candidates if roles[item[2]] == role]
            role_nearest = role_candidates[0] if role_candidates else None
            result[f"nearest_{safe_role}_probe_name"] = role_nearest[1] if role_nearest else None
            result[f"nearest_{safe_role}_probe_distance"] = role_nearest[0] if role_nearest else float("nan")
        records.append(result)
    return pd.DataFrame(records)

In [ ]:
#| export
def check_internal_compatibility(
    panel,
    *,
    min_distance: int = 75,
    role_column: str = "panel_role",
    **kwargs,
) -> pd.DataFrame:
    """Audit sequence compatibility and genomic spacing within one combined panel."""
    sequence = check_panel_compatibility(panel, panel, exclude_same_id=True, **kwargs)
    # Query-versus-reference spacing columns mean "existing" only for an
    # actual external reference panel. In a self-audit they would overwrite
    # previously attached existing-panel evidence with another add-on.
    sequence = sequence.drop(columns=[
        "nearest_existing_panel_probe_name", "nearest_existing_panel_probe_distance",
        "existing_panel_spacing_violation",
    ], errors="ignore")
    sequence["panel_compatibility_reasons"] = sequence["panel_compatibility_reasons"].map(
        lambda reasons: [reason for reason in reasons if reason != "existing_panel_spacing"]
    )
    sequence["panel_compatibility_pass"] = sequence["panel_compatibility_reasons"].map(
        lambda reasons: not bool(reasons)
    )
    sequence = sequence.rename(columns={"candidate_id": "panel_record_id"}).set_index("panel_record_id")
    spacing = nearest_probe_distances(panel, min_distance=min_distance, role_column=role_column).set_index("panel_record_id")
    result = sequence.join(spacing, how="outer")
    result["panel_sequence_compatibility_pass"] = result["panel_compatibility_pass"].astype(bool)
    result["panel_compatibility_reasons"] = [
        list(reasons) + (["panel_spacing_violation"] if bool(violation) else [])
        for reasons, violation in zip(result["panel_compatibility_reasons"], result["panel_spacing_violation"])
    ]
    result["panel_compatibility_pass"] = (
        result["panel_sequence_compatibility_pass"] & ~result["panel_spacing_violation"].astype(bool)
    )
    return result.reset_index()